# ДЗ2, опыт25: LR и dropout после нормировки эмбеддингов

**Гипотеза:** LR0.002 был выбран при старом token init N(0,1)×√d. В R5 начальные token vectors уменьшены в16 раз; лучший LR и роль dropout могли измениться. Объяснение про величину шага — гипотеза, механизм отдельно не измерен. Сохранённый R5 №24: **1602/4096 (39.11%),406/991, CE4k0.491446**; train12.95мин. Новых результатов25 пока нет.

В каждом новом запуске **6250 updates ×batch128=800000 предъявлений**,52 797 347 полезных target-токенов и38 706 084 source. Та же A4+4/d256/h4/SwiGLU680 (12086912 параметров), BPE8k№22, train500k без reference-фильтрации. AMP FP16/GradScaler1024, bucketing, fused AdamW, wd0.01, unscale→clip1, dropout0.1. Warmup313(ceil5%)→cosine до10% пика внутри6250 шагов. Дополнительных train-батчей для прогрева нет.

Начало — **точный initial-state24**, seed42; только token-table /√256, так что std≈d^-0.5 при прежнем множителе√d. Позиции и остальные начальные tensors прежние. План24 используется непосредственно из NPY с SHA; во всех новых запусках одинаковы пары и их порядок.

|Запуск|Peak LR|Dropout|Условие|
|---|---:|---:|---|
|R5, контроль24|0.002|0.1|Загрузить и переоценить|
|L1|0.001|0.1|С нуля|
|L2|0.003|0.1|С нуля|
|L3 условно|0.0005 /0.0045|0.1|Если лучший{L1,R5,L2} —0.001 /0.003; при0.002 пропустить|
|D0|Лучший LR послеL3|0|С нуля, та же инициализация/план|

**Правила до запусков:** лучший LR — exec4k; если разница<0.5п.п., выбрать поCE4k. Для однозначного выбора среди нескольких: отобрать модели, отстающие от максимума exec менее чем на0.005×4096 (максимум20 совпадений), затем минимальный CE, затемID. Разница21 уже выбирается поexec. Dropout0 принимается при потере≤0.5п.п. относительно выбранного LR. Из выбранного LR и принятогоD0 итоговый checkpoint выбирается тем же правилом exec/CE. McNemar против переоценённогоR5 — отдельная диагностика.

Оценка прежняя: greedyFP16 sortedb512/cap512; exec4k основная/old991, CEFP32, length bins reference-content безEOS. Старый heldout содержит1024 строки дляCE,991 числовую дляexec. Heldout4k точный из24; fresh1024 не оценивается. Контроль должен воспроизвести1602±3 и406±3; CE4k0.491446/CEold0.499680 с abs tolerance1e-4, иначе обучение не начинается.

Бюджет **60–75мин на однойT4**, ориентир13мин наtrain (условный повторR5 дляcheckpoint может добавить13мин); фактическое время логируется. Полные запуски не укорачиваются. Новые данные/фильтрация/modelB/sampling/beam относятся к26+.

In [ ]:
import time
if '_RECIPE25_FIRST_CELL' not in globals():_RECIPE25_FIRST_CELL=time.monotonic()
%pip install -q tokenizers==0.23.2 pyarrow sympy

## Colab, вход и resume

Выбрать однуT4. **До GPU-сессии** положить `25-start-from-24-r5-and-fixed-plan.zip` (100.08МБ) в `MyDrive/nlp-hw02/`. Архив содержит исходный initial24, весаR5, tokenizer, heldout и план; состояния оптимизатораR5 в24 не сохранялись. Вход25 принимает также полный исходныйZIP24 или готовыйZIP25 дляresume.

По умолчанию: автоматический reuse проверенной папки; иначе Drive→локальный диск. Для уже загруженных файлов выбрать `paths`, для ручного браузерного ввода — `upload`. Обучение/инференс работают с локальными файлами. В новомruntime дляresume указать ZIP25 в `INPUT_PATHS`; стартовыйZIP не содержит результатов25.

Checkpoint каждого нового запуска содержит **model/AdamW/GradScaler/scheduler/RNG** и provenance. ZIP обновляется послеtrain иevaluation. При повторе ячеек train с completion-receipt не запускается; незавершённая оценка продолжается. Если прерван самtrain, он начинается заново с общегоinitial-state; step-resume в этом notebook не включён. Послеinterrupt можно выполнить последнюю самостоятельнуюячейку.

**Особый случай:** у загруженногоR5 нет состояний AdamW/scaler. Если R5 останется выбранной базой, полноценный checkpoint можно получить повтором его рецепта с нуля; это повтор для сохранения состояния, его фактические метрики оцениваются отдельно. К старым весам не приписываются вновь созданные состояния. По умолчанию `CONTROL_STATE_POLICY='replay'`: этот повтор выполняется только если R5 выбран дляэкспорта. Можно выбрать `weights_only`, если исходныйR5 без optimizer-state предпочтительнее дополнительногоtrain; это отклонение оттребования полногоcheckpoint будетявно отраженовотчёте. Здесь повтор не запускался.

In [ ]:
import os,sys,json,hashlib,gc,subprocess,importlib
from pathlib import Path
os.environ['TOKENIZERS_PARALLELISM']='false';os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='1'
MODULE_DIR=Path('hw02-recipe25-modules');MODULE_DIR.mkdir(exist_ok=True)
MODULE_SOURCES={'recipe25_core.py': 'import ast,contextlib,gc,hashlib,json,math,random,time\n\nfrom array import array\n\nfrom pathlib import Path\n\nfrom collections import Counter\n\nimport torch\n\nfrom torch import nn\n\nfrom torch.nn import functional as F\n\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\n\nfrom torch.nn.utils.rnn import pad_sequence\n\nfrom tokenizers import Tokenizer\n\nVOCAB=8192\n\nMAX_POSITIONS=2048\n\nGENERATION_CAP=512\n\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError(\'Position overflow\')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\ndef sync(device):\n if device.type==\'cuda\':torch.cuda.synchronize(device)\n\ndef init_tokenizer(path):\n tokenizer=Tokenizer.from_file(str(path))\n if tokenizer.get_vocab_size()!=VOCAB-TOKEN_OFFSET:raise ValueError(\'Unexpected vocabulary\')\n visible=list(range(33,127))+list(range(161,173))+list(range(174,256));bchars={b:chr(b)for b in visible}\n for i,b in enumerate(b for b in range(256)if b not in bchars):bchars[b]=chr(256+i)\n charbytes={c:b for b,c in bchars.items()}\n tokenbytes=[b\'\',b\'\',b\'\']+[bytes(charbytes[c]for c in tokenizer.id_to_token(i))for i in range(VOCAB-3)]\n return tokenizer,tokenbytes\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b\'\'.join(chunks)\n\ndef source_batch(tokenizer,questions,device):\n return pad_sequence([torch.tensor([BOS]+encode(tokenizer,q)+[EOS])for q in questions],batch_first=True,padding_value=PAD).to(device)\n\ndef collate_cache(cache,indices,device):\n src=pad_sequence([torch.tensor(cache[i][0],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n tgt=pad_sequence([torch.tensor([BOS]+list(cache[i][1]),dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n labels=pad_sequence([torch.tensor(list(cache[i][1])+[EOS],dtype=torch.long)for i in indices],batch_first=True,padding_value=PAD)\n counts={\'source_tokens\':sum(len(cache[i][0])for i in indices),\'target_tokens\':sum(len(cache[i][1])+1 for i in indices),\n  \'source_slots\':src.numel(),\'target_slots\':labels.numel(),\'pairs\':len(indices)}\n return src.to(device),tgt.to(device),labels.to(device),counts\n\ndef bucket_plan(cache,batch,steps,seed=230042,chunk_pairs=4096):\n if chunk_pairs%batch:raise ValueError(\'Chunk must be divisible by batch\')\n rng=random.Random(seed);plans=[]\n while len(plans)<steps:\n  chunk=rng.sample(range(len(cache)),min(chunk_pairs,len(cache)))\n  chunk.sort(key=lambda i:(len(cache[i][0])+len(cache[i][1])+1,len(cache[i][0]),len(cache[i][1])))\n  batches=[chunk[i:i+batch]for i in range(0,len(chunk)-batch+1,batch)]\n  if not batches:raise ValueError(\'Too few data for this batch size\')\n  rng.shuffle(batches);plans.extend(batches)\n return plans[:steps]\n\ndef sdpa_flags():\n return {k:bool(getattr(torch.backends.cuda,k+\'_sdp_enabled\')())for k in [\'flash\',\'mem_efficient\',\'math\',\'cudnn\']if hasattr(torch.backends.cuda,k+\'_sdp_enabled\')}\n\ndef profile_sdpa(call,device):\n if device.type!=\'cuda\':return {\'status\':\'CPU_smoke\',\'operators\':[]}\n with torch.profiler.profile(activities=[torch.profiler.ProfilerActivity.CPU,torch.profiler.ProfilerActivity.CUDA])as prof:\n  call();sync(device)\n ops=sorted({e.key for e in prof.key_averages()if \'scaled_dot_product\'in e.key})\n return {\'status\':\'observed\',\'operators\':ops,\'scope\':\'one inference preflight outside timers; other shapes/backward may use a different backend\'}\n\ndef fused_adamw_available(device):\n if device.type!=\'cuda\':return False\n try:\n  p=nn.Parameter(torch.ones(1,device=device));opt=torch.optim.AdamW([p],lr=1e-4,fused=True);p.grad=torch.ones_like(p);opt.step();sync(device)\n  del opt,p;return True\n except (TypeError,RuntimeError):return False\n\nclass CachedDecoder:\n    @torch.inference_mode()\n    def __init__(self, model, src, capacity):\n        model.eval()\n        if not 0 < capacity <= model.positions.num_embeddings:\n            raise ValueError(\'Invalid cache capacity\')\n        self.model, self.capacity, self.position = model, capacity, 0\n        self.batch_size = len(src)\n        self.width = model.tokens.embedding_dim\n        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads\n        self.head_dim = self.width // self.heads\n        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))\n        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).\n        self.source_allowed = src.ne(PAD)[:, None, None, :]\n        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)\n        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))\n        self.layers = []\n        for layer in model.transformer.decoder.layers:\n            if not layer.norm_first:\n                raise ValueError(\'This experiment requires Pre-LN decoder layers\')\n            for attention in (layer.self_attn, layer.multihead_attn):\n                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads\n                        or attention.bias_k is not None or attention.bias_v is not None\n                        or attention.add_zero_attn):\n                    raise ValueError(\'Unsupported attention configuration\')\n            attention = layer.multihead_attn\n            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],\n                                    attention.in_proj_bias[self.width:])\n            cross_k, cross_v = projected_kv.chunk(2, dim=-1)\n            shape = (len(src), self.heads, capacity, self.head_dim)\n            self.layers.append({\n                \'self_k\': memory.new_empty(shape), \'self_v\': memory.new_empty(shape),\n                \'cross_k\': self.split_heads(cross_k), \'cross_v\': self.split_heads(cross_v),\n            })\n\n    def split_heads(self, values):\n        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)\n\n    def attend(self, attention, q, k, v, allowed):\n        # One query at the latest position; all supplied keys are from its past/current.\n        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.\n        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,\n                                                dropout_p=0.0, is_causal=False)\n        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)\n        return attention.out_proj(output)\n\n    @torch.inference_mode()\n    def step(self, token_ids):\n        if self.position >= self.capacity:\n            raise ValueError(\'Cache capacity exceeded\')\n        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:\n            raise ValueError(\'One token per batch row is required\')\n        t = self.position\n        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)\n        x = x + self.position_vectors[t][None, None, :]\n        self.valid[:, t] = token_ids.ne(PAD)\n        self_allowed = self.valid[:, None, None, :t + 1]\n        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):\n            attention = layer.self_attn\n            q, k, v = F.linear(layer.norm1(x), attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)\n            cache[\'self_k\'][:, :, t:t + 1] = self.split_heads(k)\n            cache[\'self_v\'][:, :, t:t + 1] = self.split_heads(v)\n            self_output = self.attend(attention, self.split_heads(q),\n                cache[\'self_k\'][:, :, :t + 1], cache[\'self_v\'][:, :, :t + 1], self_allowed)\n            x = x + layer.dropout1(self_output)\n\n            attention = layer.multihead_attn\n            q = F.linear(layer.norm2(x), attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])\n            cross_output = self.attend(attention, self.split_heads(q),\n                                      cache[\'cross_k\'], cache[\'cross_v\'], self.source_allowed)\n            x = x + layer.dropout2(cross_output)\n            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x)))))\n            x = x + layer.dropout3(feed_forward)\n\n        if self.model.transformer.decoder.norm is not None:\n            x = self.model.transformer.decoder.norm(x)\n        self.position += 1\n        return self.model.head(x[:, 0])\n\ndef code_properties(code):\n    def scoped_nodes(node):\n        stack = [node]\n        while stack:\n            current = stack.pop()\n            yield current\n            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):\n                stack.extend(ast.iter_child_nodes(current))\n    try:\n        tree = ast.parse(code)\n        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == \'solve\']\n        has_return = any(isinstance(n, ast.Return) and n.value is not None\n            for f in functions for stmt in f.body for n in scoped_nodes(stmt))\n        return {\'parses_as_python\': True, \'defines_solve\': bool(functions), \'solve_returns_value\': has_return,\n                \'syntax_error_type\': None, \'syntax_error_message\': None, \'indentation_error\': False}\n    except (SyntaxError, ValueError, RecursionError) as error:\n        return {\'parses_as_python\': False, \'defines_solve\': False, \'solve_returns_value\': False,\n                \'syntax_error_type\': type(error).__name__, \'syntax_error_message\': str(error),\n                \'indentation_error\': isinstance(error, IndentationError)}\n\nclass BudgetExpired(Exception):pass\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired(\'Session budget reached; completed run boundaries are preserved\')\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n@torch.inference_mode()\ndef generate_preallocated(model,src,cap=GENERATION_CAP,deadline=None,check_every=16):\n """Only output storage and termination checks change; same encoder/KV/argmax."""\n cache=CachedDecoder(model,src,cap)\n output=torch.full((len(src),cap),PAD,device=src.device,dtype=torch.long)\n token=torch.full((len(src),),BOS,device=src.device,dtype=torch.long)\n done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n for step in range(cap):\n  if step%check_every==0:check_deadline(deadline)\n  logits=cache.step(token);logits[:,[PAD,BOS]]=-torch.inf\n  token=logits.argmax(-1).masked_fill(done,PAD);output[:,step]=token;done|=token.eq(EOS)\n  # Only one device->CPU synchronization per 16 steps, plus final output transfer.\n  # Rows already at EOS remain PAD; active_tokens removes these extra slots.\n  if (step+1)%check_every==0 and bool(done.all()):break\n return output[:,:cache.position],cache.position\n\ndef restore_order(ordered_values,original_indices,size):\n if len(ordered_values)!=size or sorted(original_indices)!=list(range(size)):\n  raise ValueError(\'Not a complete permutation\')\n restored=[None]*size\n for value,i in zip(ordered_values,original_indices):restored[i]=value\n return restored\n\ndef inspect_output(row,ids,tokenizer,tokenbytes):\n active=active_tokens(ids);raw=decode_bytes(tokenbytes,active)\n try:code=raw.decode(\'utf-8\');valid=True\n except UnicodeDecodeError:code=raw.decode(\'utf-8\',errors=\'replace\');valid=False\n example={\'question\':row[\'question\'],\'generated_code\':code,\'generated_token_ids\':active,\n  \'generated_tokens_including_eos\':len(active),\'generated_utf8_bytes\':len(raw),\'valid_utf8\':valid,\n  \'eos_emitted\':EOS in active,\'reached_token_cap\':len(active)==GENERATION_CAP,\n  \'token_cap_without_eos\':len(active)==GENERATION_CAP and EOS not in active,\n  \'byte_budget_stop\':False,\'stop_reason\':\'eos\'if EOS in active else\'token_cap\',**code_properties(code)}\n if \'code\'in row:\n  gold=encode(tokenizer,row[\'code\'])+[EOS]\n  example.update(reference_code=row[\'code\'],reference_utf8_bytes=len(row[\'code\'].encode()),\n   reference_bpe_tokens_including_eos=len(gold),exact_code_match=valid and code==row[\'code\'],exact_tokens_with_eos=active==gold)\n return example\n\ndef sha_file(path):\n h=hashlib.sha256()\n with Path(path).open(\'rb\')as f:\n  for chunk in iter(lambda:f.read(1024**2),b\'\'):h.update(chunk)\n return h.hexdigest()\n\ndef write_json(path,value):\n path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_suffix(path.suffix+\'.tmp\')\n tmp.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\');tmp.replace(path)\n\nMODEL_SPEC=dict(d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680)\n', 'recipe25_exec.py': '"""Bounded numeric-program execution for this public dataset; no arbitrary imports/I/O."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef execute_task(payload):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    tree,status=filter_program(code)\n    if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:signal.setitimer(signal.ITIMER_REAL,0)\n\ndef create_pool(workers,memory_bytes):\n    return mp.get_context(\'spawn\').Pool(workers,initializer=worker_init,\n        initargs=(memory_bytes,),maxtasksperchild=256)\n\ndef evaluate_programs(codes,workers=2,wall_limit=1.0,cpu_limit=2.0,\n                      parent_timeout=3.0,memory_bytes=768*1024**2,deadline=None,progress=None):\n    """Bounded inflight tasks; every async result has a parent deadline.\n    A killed worker loses its task even if Pool replaces it: terminate/recreate\n    the WHOLE pool, retain completed results and retry collateral pending tasks.\n    Pool.map is deliberately never used. Output order and duplicates preserved.\n    """\n    if parent_timeout<=wall_limit:raise ValueError(\'Parent deadline must exceed program deadline\')\n    unique=list(dict.fromkeys(codes));todo=deque(range(len(unique)));answers={};pending={}\n    pool=None;restarts=0;start=time.monotonic();cancelled=False;last_progress=-1\n    try:\n        while todo or pending:\n            if deadline is not None and time.monotonic()>=deadline:\n                cancelled=True;break\n            if pool is None:pool=create_pool(workers,memory_bytes)\n            while todo and len(pending)<workers:\n                i=todo.popleft();tree,status=filter_program(unique[i])\n                if status:answers[i]={\'status\':status};continue\n                pending[i]=(pool.apply_async(execute_task,((unique[i],wall_limit,cpu_limit),)),time.monotonic())\n            for i,(future,submitted)in list(pending.items()):\n                if future.ready():\n                    try:answers[i]=future.get(timeout=0)\n                    except Exception as e:answers[i]={\'status\':\'worker_error\',\'error_type\':type(e).__name__}\n                    del pending[i]\n            timed=[i for i,(_,submitted)in pending.items()if time.monotonic()-submitted>=parent_timeout]\n            if timed:\n                for i in timed:answers[i]={\'status\':\'timeout_or_worker_exit\'}\n                retry=[i for i in pending if i not in timed]\n                pool.terminate();pool.join();pool=None;restarts+=1;pending.clear()\n                todo.extendleft(reversed(retry))\n            if progress and len(answers)!=last_progress and (len(answers)%1000==0 or len(answers)==len(unique)):\n                progress(len(answers),len(unique));last_progress=len(answers)\n            if pending:time.sleep(.001)\n        if cancelled:\n            for i in todo:answers[i]={\'status\':\'budget_not_evaluated\'}\n            for i in pending:answers[i]={\'status\':\'budget_not_evaluated\'}\n    finally:\n        if pool is not None:pool.terminate();pool.join()\n    by_code={c:answers.get(i,{\'status\':\'budget_not_evaluated\'})for i,c in enumerate(unique)}\n    rows=[dict(by_code[c])for c in codes]\n    return {\'rows\':rows,\'summary\':{\'programs\':len(codes),\'unique_programs\':len(unique),\n        \'statuses\':dict(Counter(r[\'status\']for r in rows)),\'pool_restarts\':restarts,\n        \'seconds\':time.monotonic()-start,\'complete\':not cancelled}}\n\ndef compare_exec(reference_results,generation_results,rel_tol=1e-6,abs_tol=1e-6):\n    refs=reference_results[\'rows\']if isinstance(reference_results,dict)else reference_results\n    gens=generation_results[\'rows\']if isinstance(generation_results,dict)else generation_results\n    if len(refs)!=len(gens):raise ValueError(\'Order/length mismatch\')\n    eligible=[i for i,r in enumerate(refs)if r[\'status\']==\'numeric\']\n    matches=[i for i in eligible if gens[i][\'status\']==\'numeric\'and math.isclose(\n        gens[i][\'value\'],refs[i][\'value\'],rel_tol=rel_tol,abs_tol=abs_tol)]\n    return {\'denominator\':len(eligible),\'matches\':len(matches),\n        \'accuracy\':len(matches)/len(eligible)if eligible else None,\n        \'eligible_indices\':eligible,\'match_indices\':matches,\n        \'reference_statuses\':dict(Counter(r[\'status\']for r in refs)),\n        \'generation_statuses\':dict(Counter(r[\'status\']for r in gens)),\n        \'rel_tol\':rel_tol,\'abs_tol\':abs_tol}\n', 'recipe25_data.py': 'import hashlib,json,random,time\nfrom pathlib import Path\nfrom array import array\nfrom urllib.request import urlretrieve\nimport pyarrow.parquet as pq\nfrom recipe25_core import sha_file,write_json,encode,BOS,EOS,check_deadline\nREVISION=\'b2724689ff079e9c22efd548fe134bfb29036dbb\'\nSHARDS=[\n {\'path\':\'data/train/part-00000.parquet\',\'bytes\':26169796,\'sha256\':\'91831d582faa12d73b1a6b3878e496446cd5df074f17ee4ffea8cc2887c708ec\'},\n {\'path\':\'data/train/part-00001.parquet\',\'bytes\':26166175,\'sha256\':\'5222d657a1af07a531f1b22503f35741a6547f0831509897d0a279bc8f67d068\'},\n {\'path\':\'data/train/part-00002.parquet\',\'bytes\':26162244,\'sha256\':\'c6082401ea966aeaa8375018ae1f1d9c7c55cc30e421c45cc933ceacd97dc3de\'}]\nTRAIN_HASH=\'61483f4dc6970ff5f2b1afffa45d57391dcfc01ab6ec4f140c6e6b6c792c8c38\'\nLOCATIONS_HASH=\'c2332a54a923ffa85da9ac91ec0a39a2e4ead7f7c2e328abecef702e1d306ba7\'\nHELDOUT_HASH=\'e187bc297218365f1a1452a15376e0ff2fceaf8a5a5517820549854b90564743\'\nTEST_HASH=\'8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a\'\n\ndef data_hash(rows):return hashlib.sha256(json.dumps(rows,ensure_ascii=False,sort_keys=True).encode()).hexdigest()\ndef is_validation(question):return int.from_bytes(hashlib.sha256(f\'42:{question}\'.encode()).digest()[:8],\'big\')%10==0\ndef eligible(row):return len(row[\'question\'].encode())<=512 and len(row[\'code\'].encode())<512\n\ndef download_checked(spec,cache,deadline=None):\n check_deadline(deadline);cache=Path(cache);cache.mkdir(parents=True,exist_ok=True);target=cache/Path(spec[\'path\']).name\n if not target.exists():\n  tmp=target.with_suffix(\'.download\')\n  # Network calls have a bounded socket timeout; deadline checked again afterward.\n  import socket\n  socket.setdefaulttimeout(30)\n  urlretrieve(f\'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{REVISION}/{spec["path"]}\',tmp)\n  if sha_file(tmp)!=spec[\'sha256\']:raise ValueError(\'Downloaded SHA mismatch\')\n  tmp.replace(target)\n if sha_file(target)!=spec[\'sha256\']:raise ValueError(\'Cached SHA mismatch\')\n if \'bytes\'in spec and target.stat().st_size!=spec[\'bytes\']:raise ValueError(\'Size mismatch\')\n check_deadline(deadline);return target\n\ndef prepare_data(cache,tokenizer,deadline=None,progress=print):\n started=time.monotonic()\n test_file=download_checked({\'path\':\'data/test/test.parquet\',\'sha256\':TEST_HASH},cache,deadline)\n test=[{\'question\':q}for q in pq.read_table(test_file,columns=[\'question\'])[\'question\'].to_pylist()]\n assert len(test)==1919;test_questions={r[\'question\']for r in test}\n train=[];locations=[];validation=None;heldout_pool=[];heldout_questions=set();paths=[];fresh=[];fresh_questions=set()\n # Exact21/22 selection. No reference-exec filtering in this throughput experiment.\n for shard_index,spec in enumerate(SHARDS):\n  path=download_checked(spec,cache,deadline);paths.append(path);offset=0\n  for batch in pq.ParquetFile(path).iter_batches(batch_size=4096):\n   check_deadline(deadline);rows=batch.to_pylist()\n   if validation is None:\n    validation=[r for r in rows if eligible(r)and is_validation(r[\'question\'])][:64]\n    assert len(validation)==64;heldout_questions.update(r[\'question\']for r in validation)\n   for i,row in enumerate(rows):\n    if not eligible(row)or row[\'question\']in test_questions:continue\n    if is_validation(row[\'question\']):\n     if shard_index==0 and row[\'question\']not in heldout_questions and len(heldout_pool)<960:\n      heldout_pool.append(row);heldout_questions.add(row[\'question\'])\n     if shard_index>0 and row[\'question\']not in heldout_questions and row[\'question\']not in fresh_questions and len(fresh)<1024:\n      fresh.append(row);fresh_questions.add(row[\'question\'])\n     continue\n    train.append(row);locations.append((shard_index,offset+i))\n    if len(train)==500000:break\n   if len(train)==500000:break\n   offset+=len(rows)\n  if progress:progress(f\'Data shard{shard_index}: selected {len(train)} /500000\')\n  if len(train)==500000:break\n heldout=validation+heldout_pool\n assert len(train)==500000 and len(heldout)==1024 and len(heldout_questions)==1024\n assert data_hash(train)==TRAIN_HASH and data_hash(locations)==LOCATIONS_HASH and data_hash(heldout)==HELDOUT_HASH\n assert not any(r[\'question\']in heldout_questions or r[\'question\']in test_questions for r in train)\n summary={\'revision\':REVISION,\'train_rows\':len(train),\'heldout_rows\':len(heldout),\'test_rows\':len(test),\n  \'train_sha256\':TRAIN_HASH,\'locations_sha256\':LOCATIONS_HASH,\'heldout_sha256\':HELDOUT_HASH,\'shards\':SHARDS,\n  \'selection_seconds\':time.monotonic()-started,\'filter_reference_non_number\':False,\'fresh_heldout_not_evaluated\':True}\n # encode_batch keeps the unchanged tokenization but avoids a Python call per string.\n token_cache=[];encoding_started=time.monotonic()\n for lo in range(0,len(train),2048):\n  check_deadline(deadline);part=train[lo:lo+2048]\n  questions=tokenizer.encode_batch([r[\'question\']for r in part],add_special_tokens=False)\n  codes=tokenizer.encode_batch([r[\'code\']for r in part],add_special_tokens=False)\n  token_cache.extend((array(\'H\',[BOS]+[i+3 for i in q.ids]+[EOS]),array(\'H\',[i+3 for i in c.ids]))for q,c in zip(questions,codes))\n summary[\'bpe_cache_seconds\']=time.monotonic()-encoding_started\n summary[\'source_tokens\']=sum(len(x[0])for x in token_cache);summary[\'target_tokens\']=sum(len(x[1])+1 for x in token_cache)\n assert len(fresh)==1024 and data_hash(fresh)==\'eaf5f7c00559d6b1e5650a0dd879e5b4a214931ee803f8c1f6efd956c54971b2\'\n summary[\'fresh_protection\']={\'questions\':1024,\'pairs_sha256\':data_hash(fresh),\'usage\':\'exclusion set only; not encoded/evaluated/saved/modified\'}\n return token_cache,heldout,test,paths,summary,fresh_questions\n', 'recipe25_run.py': '"""Experiment25: LR/dropout with normalized token initialization; fixed24 data and plan."""\nimport gc, hashlib, json, math, os, time, zipfile\nfrom collections import Counter\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nimport recipe25_core as core\nimport recipe25_exec as numeric\n\nSEED=42\nPLAN_SEED=230042\nPAIR_PRESENTATIONS=800_000\nPROTOCOL={\'experiment\': 25, \'revision\': \'b2724689ff079e9c22efd548fe134bfb29036dbb\', \'model\': {\'d_model\': 256, \'heads\': 4, \'encoder_layers\': 4, \'decoder_layers\': 4, \'swiglu_dim\': 680}, \'train_pairs\': 500000, \'pair_presentations\': 800000, \'batch\': 128, \'updates\': 6250, \'plan_seed\': 230042, \'init_seed\': 42, \'chunk_pairs\': 4096, \'initial_token_std\': \'d^-0.5; original24 token draws /sqrt(d); embedding multiplier sqrt(d); positions unchanged\', \'dropout\': 0.1, \'peak_LRs\': [0.001, 0.003], \'control_peak_LR\': 0.002, \'conditional_L3\': {\'if_best_.001\': 0.0005, \'if_best_.003\': 0.0045, \'if_best_.002\': \'skip\'}, \'D0\': \'best LR after conditional L3, dropout0; same normalized initialization/plan\', \'warmup_fraction\': 0.05, \'warmup_rounding\': \'ceil\', \'cosine_floor_fraction\': 0.1, \'weight_decay\': 0.01, \'clip\': 1.0, \'AMP\': \'FP16 autocast + GradScaler; unscale before clip\', \'initial_scale\': 1024, \'optimizer\': \'fused AdamW when supported; identical mode across runs and boundary resume\', \'best_LR_rule\': \'take candidates within <0.005 accuracy of highest exec4k; among these minimum CE4k, then stable ID\', \'dropout0_max_accuracy_drop\': 0.005, \'generation\': \'FP16 sorted batch512; preallocated output; CPU EOS check16; greedy cap512\', \'CE\': \'FP32 eval, token-weighted sum/PAD excluded, EOS included, batch32\', \'reference_length_bins\': \'content BPE tokens without EOS: [0,60),[60,90),[90,120),[120,inf)\', \'metric\': \'executed reference solve; finite numeric; bool->int; math.isclose rel=abs=1e-6\', \'McNemar\': \'exact two-sided paired versus reevaluated R5; diagnostic, not a selection gate\', \'control_expected\': {\'heldout4k_matches\': 1602, \'match_tolerance\': 3, \'CE4k\': 0.49144607270420937, \'CE_old1024\': 0.49967962335569877, \'CE_abs_tolerance\': 0.0001, \'old991_matches\': 406}, \'expected_budget_minutes\': [60, 75], \'saved_R5_training_seconds\': 776.9073420000004, \'resume\': \'completed training/evaluation receipts; interrupted training restarts from common initialization\', \'quality_eligibility\': \'all planned presentations; finite loss/grad; zero AMP skipped updates\', \'fresh1024\': \'exclusion hash verified only; never encoded/evaluated/modified\', \'train_filter_reference\': False, \'tokenizer_sha256\': \'261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d\', \'control_R5_model_sha256\': \'0459432561dc0b19605d7516062e70ef9c45a4dcbb21b8cd8ee2a59687f7de09\', \'initial_file_sha256\': \'daedd90fe51fef95dc151bc28e2f1ed0c66de80e5aa50a9ad8723ec50204c647\', \'initial_tensor_sha256\': \'7e588b942e0aab8e39873e63a6acf7d6446623db17c5be50f297a6bb4b472489\', \'normalized_initial_tensor_sha256\': \'974f9b6efca5e95e8b4e9269858984d6c70bdf62e354f490931d1b984005e380\', \'heldout4k_pairs_sha256\': \'6b0d0d559a56921faac1842e3508889225af1fe1ad8e04de378840bc4df8b047\', \'heldout4k_file_sha256\': \'28a3c0c9088c5b4a17d75d216e4abc4262cc9ca62884966c43e7ec09b6f040c2\', \'batch_plan_file_sha256\': \'48734ce4d9621e0adf464e5fad25bfc625772d7a222baeada107ea92c0a13628\', \'expected_useful_source_tokens\': 38706084, \'expected_useful_target_tokens\': 52797347, \'control_optimizer_state\': \'not available in24; never invent states for its original weights\'}\n\ndef protocol_hash():\n    return hashlib.sha256(json.dumps(PROTOCOL,sort_keys=True).encode()).hexdigest()\n\ndef save_model_state(path, model):\n    """Every PT contains only a dict of FP32 CPU model tensors. No optimizer/RNG pickle."""\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    state={k:v.detach().cpu().float().contiguous() for k,v in model.state_dict().items()}\n    tmp=path.with_suffix(\'.pt.tmp\');torch.save(state,tmp);tmp.replace(path)\n    return core.sha_file(path)\n\ndef state_digest(state):\n    h=hashlib.sha256()\n    for k,v in sorted(state.items()):\n        a=v.detach().cpu().contiguous().numpy();h.update(k.encode());h.update(str(a.shape).encode());h.update(a.tobytes())\n    return h.hexdigest()\n\ndef set_seed(seed=SEED):\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n\ndef make_initial(path, spec=None):\n    set_seed();model=core.SimpleTransformer(**(spec or core.MODEL_SPEC))\n    digest=state_digest(model.state_dict());save_model_state(path,model);del model\n    return digest\n\ndef load_model(path, device, dropout=.1, scaled_tokens=False, spec=None):\n    model=core.SimpleTransformer(**(spec or core.MODEL_SPEC),dropout=dropout)\n    payload=torch.load(path,map_location=\'cpu\',weights_only=True)\n    state=payload[\'model\'] if payload.get(\'format\')==\'nlp-hw02-training-v1\' else payload\n    model.load_state_dict(state,strict=True)\n    del payload\n    if scaled_tokens:\n        with torch.no_grad():model.tokens.weight.div_(math.sqrt(model.width))\n    del state\n    return model.to(device)\n\ndef lr_for_update(update,total,peak):\n    if not 1<=update<=total:raise ValueError(\'Update indexing is1..T\')\n    warmup=math.ceil(.05*total)\n    if update<=warmup:return peak*update/warmup\n    progress=(update-warmup)/(total-warmup)\n    return peak*(.1+.45*(1+math.cos(math.pi*progress)))\n\ndef run_spec(run_id, peak, batch=128, dropout=.1, scaled=True):\n    return {\'id\':run_id,\'peak_lr\':peak,\'batch\':batch,\'updates\':PAIR_PRESENTATIONS//batch,\n            \'dropout\':dropout,\'scaled_token_init\':scaled}\n\ndef train_one(config, initial_path, cache, common_plan, device, outdir,\n              fused=True, spec=None, amp=True, log_every=250, stop_after=None, amp_on_cpu=False):\n    """No extra warmup presentations. The LR warmup is inside the fixed plan.\n    Skipped AMP updates never cause extra presentations; the run is ineligible\n    for decisions if even one skip/nonfinite step occurred.\n    """\n    outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n    plan=common_plan.reshape(-1,config[\'batch\'])\n    assert plan.shape[0]==config[\'updates\'] and plan.size==config[\'updates\']*config[\'batch\']\n    model=opt=scaler=scheduler=None;started=time.perf_counter();train_start=train_end=None;counts=Counter();history=[]\n    summary={\'config\':config,\'status\':\'training\',\'steps_presented\':0,\'optimizer_updates\':0,\n             \'AMP_skips\':0,\'finite_loss_and_grads\':True,\'fused_adamw\':fused,\n             \'warmup_steps\':math.ceil(.05*config[\'updates\']),\n             \'initial_checkpoint_sha256\':core.sha_file(initial_path)}\n    try:\n        gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n        model=load_model(initial_path,device,dropout=config[\'dropout\'],\n                         scaled_tokens=config[\'scaled_token_init\'],spec=spec)\n        summary[\'initial_model_tensor_sha256\']=state_digest(model.state_dict())\n        if spec is None:assert summary[\'initial_model_tensor_sha256\']==PROTOCOL[\'normalized_initial_tensor_sha256\']\n        model.train();set_seed()\n        opt=torch.optim.AdamW(model.parameters(),lr=config[\'peak_lr\'],weight_decay=.01,\n                              **({\'fused\':True} if fused else {}))\n        amp_enabled=amp and (device.type==\'cuda\' or amp_on_cpu)\n        summary[\'precision\']=\'AMP_FP16\' if amp_enabled else \'FP32_CPU_smoke\'\n        scaler=torch.amp.GradScaler(device.type,enabled=amp_enabled,init_scale=1024)\n        if amp_enabled and not hasattr(scaler,\'_found_inf_per_device\'):\n            raise RuntimeError(\'Exact AMP skip diagnostic unavailable\')\n        scheduler=WarmupCosineSchedule(opt,len(plan),config[\'peak_lr\'])\n        core.sync(device);train_start=time.perf_counter()\n        for lo in range(0,len(plan),25):\n            records=[]\n            for j in range(lo,min(lo+25,len(plan))):\n                src,tgt,labels,batch_counts=core.collate_cache(cache,plan[j],device)\n                opt.zero_grad(set_to_none=True)\n                lr=scheduler.apply(j+1)\n                with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=amp_enabled):\n                    logits=model(src,tgt)\n                    loss=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=core.PAD)\n                if amp_enabled:\n                    scaler.scale(loss).backward();scaler.unscale_(opt)\n                    found=scaler._found_inf_per_device(opt)\n                    if len(found)!=1:raise RuntimeError(\'Expected one-device GradScaler state\')\n                    skipped=next(iter(found.values())).detach()\n                else:\n                    loss.backward();skipped=loss.new_zeros(())\n                norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n                finite=torch.isfinite(loss.detach()) & torch.isfinite(norm.detach())\n                records.append(torch.stack([loss.detach().float(),finite.float(),skipped.float()]))\n                if amp_enabled:scaler.step(opt);scaler.update()\n                else:opt.step()\n                counts.update(batch_counts);summary[\'steps_presented\']=j+1\n                del src,tgt,labels,logits,loss,norm\n                if stop_after is not None and j+1==stop_after:raise KeyboardInterrupt(\'CPU interruption fixture\')\n            core.sync(device);records=torch.stack(records).cpu().tolist()\n            summary[\'AMP_skips\']+=sum(int(x[2]>0) for x in records)\n            summary[\'optimizer_updates\']=summary[\'steps_presented\']-summary[\'AMP_skips\']\n            summary[\'finite_loss_and_grads\'] &= all(bool(x[1]) for x in records)\n            if not summary[\'finite_loss_and_grads\'] or summary[\'AMP_skips\']:\n                raise FloatingPointError(\'Nonfinite loss/grad or skipped AMP update; equal-update comparison invalid\')\n            if summary[\'steps_presented\']%log_every==0 or summary[\'steps_presented\']==len(plan):\n                elapsed=time.perf_counter()-train_start\n                point={\'update\':summary[\'steps_presented\'],\'lr\':lr,\'CE_last_chunk\':sum(x[0] for x in records)/len(records),\n                       \'seconds\':elapsed,\'pairs_per_second\':counts[\'pairs\']/elapsed,\n                       \'useful_target_tokens_per_second\':counts[\'target_tokens\']/elapsed}\n                history.append(point);print(config[\'id\'],json.dumps(point),flush=True)\n                core.write_json(outdir/\'progress.json\',{\'training\':summary,\'counts\':dict(counts),\'history\':history})\n        core.sync(device);train_end=time.perf_counter();summary[\'status\']=\'trained\'\n        summary[\'model_tensor_sha256\']=state_digest(model.state_dict())\n        summary[\'model_sha256\']=save_training_checkpoint(outdir/\'checkpoint.pt\',model,opt,scaler,scheduler,config,summary,spec,common_plan)\n        summary[\'checkpoint_sha256\']=summary[\'model_sha256\']\n        summary[\'checkpoint_file\']=\'checkpoint.pt\'\n        summary[\'checkpoint_states\']=[\'model\',\'AdamW\',\'GradScaler\',\'scheduler\',\'RNG\']\n        summary[\'quality_eligible\']=True\n    except (torch.cuda.OutOfMemoryError,FloatingPointError) as error:\n        summary.update(status=\'invalid_training\',quality_eligible=False,error=str(error)[:500])\n    finally:\n        core.sync(device)\n        elapsed=(train_end or time.perf_counter())-train_start if train_start is not None else 0.\n        summary.update(train_seconds=elapsed,wall_seconds=time.perf_counter()-started,counts=dict(counts),history=history,\n                       pairs_per_second=counts[\'pairs\']/elapsed if elapsed else None,\n                       useful_source_tokens_per_second=counts[\'source_tokens\']/elapsed if elapsed else None,\n                       useful_target_tokens_per_second=counts[\'target_tokens\']/elapsed if elapsed else None,\n                       peak_allocated_bytes=torch.cuda.max_memory_allocated(device) if device.type==\'cuda\' else None,\n                       peak_reserved_bytes=torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None)\n        summary[\'time_fraction_of_saved_R5_training\']=elapsed/PROTOCOL[\'saved_R5_training_seconds\']\n        if counts[\'source_slots\'] and counts[\'target_slots\']:\n            summary[\'padding_fraction\']=1-(counts[\'source_tokens\']+counts[\'target_tokens\'])/(counts[\'source_slots\']+counts[\'target_slots\'])\n        summary[\'history\']=history\n        if summary[\'status\']==\'trained\':\n            assert counts[\'pairs\']==plan.size and summary[\'optimizer_updates\']==len(plan)\n            core.write_json(outdir/\'training.json\',summary)\n        else:\n            core.write_json(outdir/\'incomplete-training.json\',summary)\n        del model,opt,scaler,scheduler;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n    return summary\n\n@torch.inference_mode()\ndef ce_one(model_path,rows,tokenizer,device,spec=None,batch=32):\n    started=time.perf_counter();model=load_model(model_path,device,spec=spec).eval();total=torch.zeros((),device=device);denominator=0\n    cached=[([core.BOS]+core.encode(tokenizer,r[\'question\'])+[core.EOS],core.encode(tokenizer,r[\'code\'])) for r in rows]\n    order=sorted(range(len(rows)),key=lambda i:len(cached[i][0])+len(cached[i][1]))\n    try:\n        for lo in range(0,len(order),batch):\n            src,tgt,labels,counts=core.collate_cache(cached,order[lo:lo+batch],device)\n            logits=model(src,tgt)\n            total+=F.cross_entropy(logits.reshape(-1,model.vocab),labels.reshape(-1),ignore_index=core.PAD,reduction=\'sum\')\n            denominator+=counts[\'target_tokens\']\n            del src,tgt,labels,logits\n        value=float(total.cpu())/denominator\n        if not math.isfinite(value):raise FloatingPointError(\'CE nonfinite\')\n        return {\'CE\':value,\'useful_target_tokens\':denominator,\'rows\':len(rows),\'precision\':\'FP32\',\'batch\':batch,\n                \'seconds\':time.perf_counter()-started}\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\n@torch.inference_mode()\ndef generate_one(model_path,rows,tokenizer_path,device,outdir,name,spec=None,batch=512,cap=512,half=True):\n    """Full timer begins before loading model tensors; ends at answers-file close."""\n    gc.collect()\n    if device.type==\'cuda\':torch.cuda.empty_cache();torch.cuda.reset_peak_memory_stats(device)\n    start=time.perf_counter();model=load_model(model_path,device,spec=spec).eval()\n    tokenizer,tokenbytes=core.init_tokenizer(tokenizer_path)\n    if half:model.half()\n    try:\n        core.generate_preallocated(model,core.source_batch(tokenizer,[rows[0][\'question\']],device),cap=min(8,cap))\n        core.sync(device);generation_start=time.perf_counter()\n        encoded=[[core.BOS]+core.encode(tokenizer,r[\'question\'])+[core.EOS] for r in rows]\n        order=sorted(range(len(rows)),key=lambda i:(len(encoded[i]),i));ordered=[];steps_sum=row_steps_sum=0\n        for lo in range(0,len(order),batch):\n            indices=order[lo:lo+batch]\n            src=nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in indices],batch_first=True,padding_value=core.PAD).to(device)\n            tokens,steps=core.generate_preallocated(model,src,cap=cap)\n            steps_sum+=steps;row_steps_sum+=steps*len(indices)\n            for i,ids in zip(indices,tokens.cpu().tolist()):\n                active=core.active_tokens(ids)\n                ordered.append({\'id\':i,\'question\':rows[i][\'question\'],\'generated_token_ids\':active,\n                    \'generated_code\':core.decode_bytes(tokenbytes,active).decode(\'utf-8\',errors=\'replace\'),\n                    \'eos_emitted\':core.EOS in active})\n        examples=core.restore_order(ordered,order,len(rows));core.sync(device)\n        generation_seconds=time.perf_counter()-generation_start\n        outdir=Path(outdir);outdir.mkdir(parents=True,exist_ok=True)\n        answers=outdir/f\'{name}-answers.jsonl\'\n        with answers.open(\'w\',encoding=\'utf-8\') as f:\n            for e in examples:f.write(json.dumps({\'id\':e[\'id\'],\'question\':e[\'question\'],\'code\':e[\'generated_code\']},ensure_ascii=False)+\'\\n\')\n        full_seconds=time.perf_counter()-start\n        core.write_json(outdir/f\'{name}-generations.json\',examples)\n        timing={\'full_seconds\':full_seconds,\'generation_seconds\':generation_seconds,\'rows\':len(rows),\n                \'decode_steps_sum\':steps_sum,\'decode_row_steps_sum\':row_steps_sum,\n                \'answers_sha256\':core.sha_file(answers),\n                \'peak_reserved_bytes\':torch.cuda.max_memory_reserved(device) if device.type==\'cuda\' else None,\n                \'timer\':\'model state load,restore,tokenizer,FP16/H2D,warmup,sorted greedy/order restore/answers write; download/cache/exec/CE/JSON dump outside\'}\n        timing[\'model_sha256\']=core.sha_file(model_path)\n        timing[\'tokenizer_sha256\']=core.sha_file(tokenizer_path)\n        return examples,timing\n    finally:\n        del model;gc.collect()\n        if device.type==\'cuda\':torch.cuda.empty_cache()\n\ndef paired_mcnemar(score,control):\n    assert score[\'eligible_indices\']==control[\'eligible_indices\']\n    s,c=set(score[\'match_indices\']),set(control[\'match_indices\']);g,l=len(s-c),len(c-s);n=g+l\n    p=min(1.,(2*sum(math.comb(n,i) for i in range(min(g,l)+1)))/(1<<n)) if n else 1.\n    return {\'gains\':g,\'losses\':l,\'discordant\':n,\'exact_two_sided_p\':p,\n            \'delta_percentage_points\':100*(g-l)/score[\'denominator\']}\n\ndef length_accuracy(rows,tokenizer,score):\n    lengths=[len(core.encode(tokenizer,r[\'code\'])) for r in rows];eligible=set(score[\'eligible_indices\']);matched=set(score[\'match_indices\'])\n    groups={k:[] for k in [\'<60\',\'60–90\',\'90–120\',\'≥120\']}\n    for i,n in enumerate(lengths):\n        if i in eligible:groups[\'<60\' if n<60 else \'60–90\' if n<90 else \'90–120\' if n<120 else \'≥120\'].append(i)\n    return {k:{\'denominator\':len(ids),\'matches\':sum(i in matched for i in ids),\n               \'accuracy\':sum(i in matched for i in ids)/len(ids) if ids else None} for k,ids in groups.items()}\n\ndef evaluate_one(model_path,config,sets,refs,tokenizer_path,device,outdir,control=None,spec=None,\n                 gen_fn=generate_one,ce_fn=ce_one,execute_fn=numeric.evaluate_programs):\n    started=time.perf_counter();outdir=Path(outdir);metrics={\'id\':config[\'id\'],\'config\':config,\'status\':\'evaluation_started\',\'splits\':{}}\n    tokenizer,_=core.init_tokenizer(tokenizer_path)\n    for split,rows in sets.items():\n        gen_file=outdir/f\'{split}-generations.json\';time_file=outdir/f\'{split}-generation-time.json\'\n        if gen_file.exists() and time_file.exists():\n            examples=json.loads(gen_file.read_text());timing=json.loads(time_file.read_text())\n            assert len(examples)==len(rows) and all(a[\'question\']==b[\'question\'] for a,b in zip(examples,rows))\n            assert timing[\'model_sha256\']==core.sha_file(model_path) and timing[\'tokenizer_sha256\']==core.sha_file(tokenizer_path)\n            assert timing[\'answers_sha256\']==core.sha_file(outdir/f\'{split}-answers.jsonl\')\n        else:\n            examples,timing=gen_fn(model_path,rows,tokenizer_path,device,outdir,split,spec=spec)\n            core.write_json(time_file,timing)\n        execution=execute_fn([e[\'generated_code\'] for e in examples])\n        assert execution[\'summary\'][\'complete\']\n        core.write_json(outdir/f\'{split}-execution.json\',execution)\n        score=numeric.compare_exec(refs[split],execution)\n        assert score[\'denominator\']==(4096 if split==\'heldout4k\' else 991)\n        ce=ce_fn(model_path,rows,tokenizer,device,spec=spec)\n        m={\'exec\':score,\'CE\':ce,\'generation_time\':timing,\'length_accuracy\':length_accuracy(rows,tokenizer,score),\n           \'execution_seconds\':execution[\'summary\'][\'seconds\']}\n        if control:m[\'McNemar_against_R5\']=paired_mcnemar(score,control[\'splits\'][split][\'exec\'])\n        metrics[\'splits\'][split]=m\n        core.write_json(outdir/\'evaluation-progress.json\',metrics)\n        print(config[\'id\'],split,\'exec\',score[\'matches\'],\'/\',score[\'denominator\'],\'CE\',ce[\'CE\'],flush=True)\n    metrics[\'status\']=\'evaluated\';metrics[\'model_sha256\']=core.sha_file(model_path)\n    metrics[\'evaluation_protocol_sha256\']=protocol_hash()\n    metrics[\'evaluation_wall_seconds_current_attempt\']=time.perf_counter()-started\n    core.write_json(outdir/\'metrics.json\',metrics)\n    return metrics\n\nclass WarmupCosineSchedule:\n    """Explicit1-based schedule with the same LR arithmetic as24; serializable."""\n    def __init__(self,optimizer,total,peak):\n        self.optimizer=optimizer;self.total=int(total);self.peak=float(peak);self.last_update=0\n    def apply(self,update):\n        if update!=self.last_update+1:raise ValueError(\'Scheduler update sequence changed\')\n        lr=lr_for_update(update,self.total,self.peak)\n        for group in self.optimizer.param_groups:group[\'lr\']=lr\n        self.last_update=update\n        return lr\n    def state_dict(self):\n        return {\'format\':\'warmup-cosine-explicit-v1\',\'total_updates\':self.total,\'peak_lr\':self.peak,\n                \'warmup_steps\':math.ceil(.05*self.total),\'floor_fraction\':.1,\'last_update\':self.last_update}\n    def load_state_dict(self,state):\n        if state[\'format\']!=\'warmup-cosine-explicit-v1\' or state[\'total_updates\']!=self.total or state[\'peak_lr\']!=self.peak:\n            raise ValueError(\'Scheduler configuration changed\')\n        if state[\'warmup_steps\']!=math.ceil(.05*self.total) or state[\'floor_fraction\']!=.1:raise ValueError(\'Schedule changed\')\n        if not 0<=state[\'last_update\']<=self.total:raise ValueError(\'Invalid schedule progress\')\n        self.last_update=int(state[\'last_update\'])\n        if self.last_update:\n            for group in self.optimizer.param_groups:group[\'lr\']=lr_for_update(self.last_update,self.total,self.peak)\n\ndef to_cpu(value):\n    if isinstance(value,torch.Tensor):return value.detach().cpu().contiguous()\n    if isinstance(value,dict):return {k:to_cpu(v) for k,v in value.items()}\n    if isinstance(value,list):return [to_cpu(v) for v in value]\n    if isinstance(value,tuple):return tuple(to_cpu(v) for v in value)\n    return value\n\ndef rng_state():\n    import random\n    np_state=np.random.get_state()\n    return {\'torch_cpu\':torch.get_rng_state(),\'torch_cuda\':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],\n            \'python\':random.getstate(),\'numpy\':(np_state[0],np_state[1].tolist(),int(np_state[2]),int(np_state[3]),float(np_state[4]))}\n\ndef restore_rng_state(state,device):\n    import random\n    torch.set_rng_state(state[\'torch_cpu\'])\n    if device.type==\'cuda\':torch.cuda.set_rng_state_all(state[\'torch_cuda\'])\n    random.setstate(state[\'python\'])\n    n=state[\'numpy\'];np.random.set_state((n[0],np.asarray(n[1],dtype=np.uint32),n[2],n[3],n[4]))\n\ndef save_training_checkpoint(path,model,opt,scaler,scheduler,config,summary,spec,plan):\n    """Self-contained model/AdamW/scaler/scheduler/RNG; weights_only-safe."""\n    state=to_cpu(model.state_dict())\n    payload={\'format\':\'nlp-hw02-training-v1\',\'experiment\':25,\'protocol_sha256\':protocol_hash(),\n        \'model\':state,\'optimizer\':to_cpu(opt.state_dict()),\'scaler\':scaler.state_dict(),\n        \'scaler_enabled\':scaler.is_enabled(),\'scheduler\':scheduler.state_dict(),\'rng\':to_cpu(rng_state()),\n        \'config\':config,\'model_spec\':dict(spec or core.MODEL_SPEC),\'fused_adamw\':summary[\'fused_adamw\'],\n        \'steps\':summary[\'optimizer_updates\'],\'model_tensor_sha256\':state_digest(state),\n        \'initial_file_sha256\':summary[\'initial_checkpoint_sha256\'],\n        \'normalized_initial_tensor_sha256\':summary[\'initial_model_tensor_sha256\'],\n        \'tokenizer_sha256\':PROTOCOL[\'tokenizer_sha256\'],\n        \'plan_tensor_sha256\':hashlib.sha256(np.asarray(plan,dtype=np.int32).tobytes()).hexdigest(),\n        \'purpose\':\'Colab_T4_training\' if next(model.parameters()).is_cuda else \'tiny_CPU_smoke\'}\n    path=Path(path);pending=path.with_suffix(\'.pt.tmp\');torch.save(payload,pending);pending.replace(path)\n    return core.sha_file(path)\n\ndef restore_training_checkpoint(path,device):\n    """Restore all states for audit/future continuation. This notebook resumes at run boundaries."""\n    payload=torch.load(path,map_location=\'cpu\',weights_only=True)\n    assert payload[\'format\']==\'nlp-hw02-training-v1\' and payload[\'experiment\']==25\n    assert payload[\'protocol_sha256\']==protocol_hash()\n    config=payload[\'config\'];model=core.SimpleTransformer(**payload[\'model_spec\'],dropout=config[\'dropout\']).to(device)\n    model.load_state_dict(payload[\'model\'],strict=True)\n    assert state_digest(model.state_dict())==payload[\'model_tensor_sha256\']\n    opt=torch.optim.AdamW(model.parameters(),lr=config[\'peak_lr\'],weight_decay=.01,\n                          **({\'fused\':True} if payload[\'fused_adamw\'] else {}))\n    opt.load_state_dict(payload[\'optimizer\'])\n    scaler=torch.amp.GradScaler(device.type,enabled=payload[\'scaler_enabled\'],init_scale=1024)\n    scaler.load_state_dict(payload[\'scaler\'])\n    scheduler=WarmupCosineSchedule(opt,config[\'updates\'],config[\'peak_lr\']);scheduler.load_state_dict(payload[\'scheduler\'])\n    assert scheduler.last_update==payload[\'steps\']\n    restore_rng_state(payload[\'rng\'],device)\n    model.train()\n    return model,opt,scaler,scheduler,payload\n\ndef select_best(metrics,ids=None):\n    """Deterministic multiway reading of \'<0.5 pp → CE\'; prevents pairwise cycles."""\n    candidates=[m for k,m in metrics.items() if (ids is None or k in ids) and m.get(\'status\')==\'evaluated\'\n                and m.get(\'quality_eligible\',True)]\n    if not candidates:raise RuntimeError(\'No complete eligible runs; see saved progress/ZIP\')\n    assert all(m[\'splits\'][\'heldout4k\'][\'exec\'][\'denominator\']==4096 for m in candidates)\n    maximum=max(m[\'splits\'][\'heldout4k\'][\'exec\'][\'matches\'] for m in candidates)\n    tied=[m for m in candidates if maximum-m[\'splits\'][\'heldout4k\'][\'exec\'][\'matches\']<.005*4096]\n    return min(tied,key=lambda m:(m[\'splits\'][\'heldout4k\'][\'CE\'][\'CE\'],m[\'id\']))\n\ndef conditional_L3(initial_best):\n    lr=initial_best[\'config\'][\'peak_lr\']\n    if lr==.001:return run_spec(\'L3\',.0005)\n    if lr==.003:return run_spec(\'L3\',.0045)\n    if lr==.002:return None\n    raise ValueError(\'Conditional L3 requires winner of {L1, R5, L2}\')\n\ndef decisions(metrics,lr_best):\n    d0=metrics[\'D0\'];base=lr_best[\'splits\'][\'heldout4k\'][\'exec\'];trial=d0[\'splits\'][\'heldout4k\'][\'exec\']\n    delta=trial[\'matches\']-base[\'matches\'];accepted=delta>=-.005*4096\n    eligible={lr_best[\'id\']:lr_best}\n    if accepted:eligible[\'D0\']=d0\n    final=select_best(eligible)\n    return {\'best_LR_run\':lr_best[\'id\'],\'best_peak_LR\':lr_best[\'config\'][\'peak_lr\'],\n        \'LR_rule\':PROTOCOL[\'best_LR_rule\'],\'dropout0\':{\'accepted\':accepted,\'delta_percentage_points\':100*delta/4096,\n            \'McNemar_against_best_LR\':paired_mcnemar(trial,base)},\n        \'best_run\':final[\'id\'],\'best_model_dropout\':final[\'config\'][\'dropout\'],\n        \'interpretation\':\'Threshold acceptance is not statistical significance; one seed and selection on4k. Fresh untouched.\'}\n\ndef verify_control(control):\n    expected=PROTOCOL[\'control_expected\'];s=control[\'splits\']\n    for split,matches,ce in [(\'heldout4k\',expected[\'heldout4k_matches\'],expected[\'CE4k\']),\n                              (\'old1024\',expected[\'old991_matches\'],expected[\'CE_old1024\'])]:\n        assert abs(s[split][\'exec\'][\'matches\']-matches)<=expected[\'match_tolerance\'],f\'R5 {split} exec reproduction failed; stop before training\'\n        assert abs(s[split][\'CE\'][\'CE\']-ce)<=expected[\'CE_abs_tolerance\'],f\'R5 {split} CE reproduction failed; stop before training\'\n    return {\'exec_tolerance\':3,\'CE_abs_tolerance\':expected[\'CE_abs_tolerance\'],\'status\':\'passed\'}\n\ndef pack_results(folder,archive):\n    folder=Path(folder);folder.mkdir(exist_ok=True)\n    files=[p for p in sorted(folder.rglob(\'*\')) if p.is_file() and p.name!=\'manifest.json\' and not p.name.endswith(\'.tmp\')]\n    manifest={\'experiment\':25,\'files\':[{\'file\':str(p.relative_to(folder)),\'bytes\':p.stat().st_size,\'sha256\':core.sha_file(p)} for p in files]}\n    core.write_json(folder/\'manifest.json\',manifest)\n    archive=Path(archive);pending=archive.with_suffix(\'.zip.tmp\')\n    with zipfile.ZipFile(pending,\'w\') as z:\n        for p in files+[folder/\'manifest.json\']:\n            z.write(p,arcname=str(p.relative_to(folder)),compress_type=zipfile.ZIP_STORED if p.suffix==\'.pt\' else zipfile.ZIP_DEFLATED,compresslevel=1)\n    with zipfile.ZipFile(pending) as z:assert z.testzip() is None\n    pending.replace(archive)\n    return {\'name\':archive.name,\'bytes\':archive.stat().st_size,\'sha256\':core.sha_file(archive)}\n\ndef completed_training(outdir,config,initial_sha):\n    folder=Path(outdir);receipt=folder/\'training.json\';weights=folder/\'checkpoint.pt\'\n    if not receipt.exists():return None\n    row=json.loads(receipt.read_text())\n    assert row[\'config\']==config and row[\'initial_checkpoint_sha256\']==initial_sha\n    assert row[\'status\']==\'trained\' and row[\'quality_eligible\'] and row[\'AMP_skips\']==0 and row[\'finite_loss_and_grads\']\n    assert row[\'checkpoint_sha256\']==row[\'model_sha256\']==core.sha_file(weights)\n    assert row[\'steps_presented\']==row[\'optimizer_updates\']==config[\'updates\'] and row[\'counts\'][\'pairs\']==PAIR_PRESENTATIONS\n    return row\n\ndef launch(config,root,initial_path,cache,plan,sets,refs,tokenizer_path,device,control,fused,\n           train_fn=train_one,eval_fn=evaluate_one,pack_fn=pack_results):\n    folder=Path(root)/\'runs\'/config[\'id\'];folder.mkdir(parents=True,exist_ok=True)\n    receipt=completed_training(folder,config,core.sha_file(initial_path))\n    if receipt is None:\n        receipt=train_fn(config,initial_path,cache,plan,device,folder,fused=fused)\n        pack_fn(root,Path(root).with_suffix(\'.zip\'))\n        if not receipt.get(\'quality_eligible\'):\n            raise RuntimeError(f"{config[\'id\']}: training invalid; do not select an LR with missing trials")\n    metrics_file=folder/\'metrics.json\';model_path=folder/\'checkpoint.pt\'\n    if metrics_file.exists():\n        metrics=json.loads(metrics_file.read_text())\n        assert metrics[\'config\']==config and metrics[\'model_sha256\']==core.sha_file(model_path)\n        assert metrics[\'evaluation_protocol_sha256\']==protocol_hash()\n    else:\n        metrics=eval_fn(model_path,config,sets,refs,tokenizer_path,device,folder,control=control)\n    metrics.update(quality_eligible=True,training=receipt)\n    core.write_json(metrics_file,metrics)\n    pack_fn(root,Path(root).with_suffix(\'.zip\'))\n    return metrics\n', 'recipe25_input.py': '"""Experiment25: exact24 R5/initial/plan/heldout and verified boundary resume."""\nimport hashlib,json,shutil,zipfile\nfrom pathlib import Path,PurePosixPath\nimport numpy as np\nimport torch\nimport recipe25_core as core\nimport recipe25_data as data\nimport recipe25_run as run\n\nORIGINAL24_SHA=\'0a184d0a86a34098d2313926a2f83448e21c5972abd55fecc96a28a8cfebaae1\'\nSTARTER_SHA=\'9cb672cb3535aa4d37482da157f60cd3986bffe3127fea6e5a92cc00d73f21b0\'\nSOURCE_MEMBERS={name:name for name in [\'initial.pt\',\'initial.json\',\'tokenizer.json\',\'batch-plan.npy\',\'plan.json\',\n \'old1024.json\',\'old1024-reference-execution.json\',\'heldout4k.json\',\'heldout4k-selection.json\',\'heldout4k-reference-execution.json\']}\nSOURCE_MEMBERS.update({\'runs/R5/model.pt\':\'controls/R5/model.pt\',\n \'runs/R5/training.json\':\'controls/R5/source-training24.json\',\'runs/R5/metrics.json\':\'controls/R5/source-metrics24.json\'})\n\nROOT_NAMES={\'starter.json\',\'protocol.json\',\'results.json\',\'decision.json\',\'manifest.json\',\'tokenizer.json\',\n \'initial.pt\',\'initial.json\',\'batch-plan.npy\',\'plan.json\',\'heldout4k.json\',\'heldout4k-reference-execution.json\',\n \'heldout4k-selection.json\',\'old1024.json\',\'old1024-reference-execution.json\',\'data.json\',\'runtime.json\',\n \'test-generations.json\',\'test-answers.jsonl\',\'test-time.json\',\'best-checkpoint.json\',\'control-reproduction.json\'}\nRUN_NAMES={\'checkpoint.pt\',\'training.json\',\'incomplete-training.json\',\'progress.json\',\'metrics.json\',\'evaluation-progress.json\',\n \'heldout4k-generations.json\',\'heldout4k-answers.jsonl\',\'heldout4k-generation-time.json\',\'heldout4k-execution.json\',\n \'old1024-generations.json\',\'old1024-answers.jsonl\',\'old1024-generation-time.json\',\'old1024-execution.json\'}\nCONTROL_NAMES=(RUN_NAMES-{\'checkpoint.pt\'})|{\'model.pt\',\'source-training24.json\',\'source-metrics24.json\'}\n\ndef allowed_name(name):\n p=PurePosixPath(name)\n if p.is_absolute() or \'..\'in p.parts or \'\\\\\'in name:return False\n if len(p.parts)==1:return name in ROOT_NAMES\n if len(p.parts)==3 and p.parts[0]==\'runs\' and p.parts[1]in {\'L1\',\'L2\',\'L3\',\'D0\',\'R5_replay\'}:return p.name in RUN_NAMES\n if len(p.parts)==3 and p.parts[:2]==(\'controls\',\'R5\'):return p.name in CONTROL_NAMES\n return False\n\ndef verify_manifest(z,experiment,kind=None):\n if z.testzip()is not None:raise ValueError(\'ZIP CRC failed\')\n m=json.loads(z.read(\'manifest.json\'));entries=m[\'files\'];names=[e[\'file\']for e in entries]\n if m.get(\'experiment\')!=experiment or (kind and m.get(\'kind\')!=kind):raise ValueError(\'Unexpected archive type\')\n if len(names)>250 or len(names)!=len(set(names)):raise ValueError(\'Unexpected manifest\')\n if set(z.namelist())!=set(names)|{\'manifest.json\'} or len(z.namelist())!=len(names)+1:raise ValueError(\'ZIP/manifest mismatch\')\n if sum(z.getinfo(n).file_size for n in names)>1_500_000_000:raise ValueError(\'Oversized archive\')\n for e in entries:\n  if not allowed_name(e[\'file\']) or not 0<=e[\'bytes\']<=200_000_000:raise ValueError(\'Unexpected archive member\')\n  raw=z.read(e[\'file\'])\n  if len(raw)!=e[\'bytes\'] or hashlib.sha256(raw).hexdigest()!=e[\'sha256\']:raise ValueError(\'Member SHA mismatch\')\n return entries\n\ndef early_placeholder(path):\n try:return Path(path).name==\'results.json\' and json.loads(Path(path).read_text())=={\'experiment\':25,\'status\':\'partial_before_setup\'}\n except (FileNotFoundError,ValueError):return False\n\ndef extract_checked(z,mapping,root):\n # Validate conflicts before writing any member; preserve completed local work.\n for source,target in mapping.items():\n  raw=z.read(source);p=Path(root)/target\n  if p.exists() and core.sha_file(p)!=hashlib.sha256(raw).hexdigest() and not early_placeholder(p):raise ValueError(\'Local/resume conflict; preserve both sessions\')\n for source,target in mapping.items():\n  p=Path(root)/target;p.parent.mkdir(parents=True,exist_ok=True)\n  if not p.exists() or early_placeholder(p):\n   with z.open(source)as f,p.open(\'wb\')as out:shutil.copyfileobj(f,out)\n\ndef validate_inputs(root,check_tensors=False):\n root=Path(root);p=run.PROTOCOL\n for name,expected in [(\'tokenizer.json\',p[\'tokenizer_sha256\']),(\'initial.pt\',p[\'initial_file_sha256\']),\n                        (\'controls/R5/model.pt\',p[\'control_R5_model_sha256\']),(\'batch-plan.npy\',p[\'batch_plan_file_sha256\']),\n                        (\'heldout4k.json\',p[\'heldout4k_file_sha256\'])]:\n  if not (root/name).exists() or core.sha_file(root/name)!=expected:raise ValueError(f\'Need exact24 input: {name}\')\n old=json.loads((root/\'old1024.json\').read_text());held=json.loads((root/\'heldout4k.json\').read_text())\n assert data.data_hash(old)==data.HELDOUT_HASH and len(old)==1024\n assert data.data_hash(held)==p[\'heldout4k_pairs_sha256\'] and len(held)==4096\n assert len({r[\'question\']for r in held})==4096 and not {r[\'question\']for r in held}&{r[\'question\']for r in old}\n plan=np.load(root/\'batch-plan.npy\',allow_pickle=False)\n assert plan.dtype==np.int32 and plan.shape==(6250,128) and 0<=plan.min()<=plan.max()<500000\n if check_tensors:\n  initial=torch.load(root/\'initial.pt\',map_location=\'cpu\',weights_only=True)\n  assert run.state_digest(initial)==p[\'initial_tensor_sha256\']\n  initial[\'tokens.weight\']=initial[\'tokens.weight\']/256**.5\n  assert run.state_digest(initial)==p[\'normalized_initial_tensor_sha256\'];del initial\n  control=run.load_model(root/\'controls/R5/model.pt\',torch.device(\'cpu\'))\n  assert all(torch.isfinite(v).all()for v in control.state_dict().values());del control\n return {\'initial_and_R5_and_tokenizer_and_plan_SHA\':\'passed\',\'heldout_exact24\':\'passed\',\n         \'tensor_checks\':check_tensors,\'fresh_usage\':\'not evaluated/encoded/modified\'}\n\ndef import_inputs(paths,root,code_sha):\n root=Path(root);root.mkdir(exist_ok=True);reports=[]\n for path in map(Path,paths):\n  sha=core.sha_file(path)\n  with zipfile.ZipFile(path)as z:\n   if sha==ORIGINAL24_SHA:\n    if z.testzip()is not None:raise ValueError(\'Original24 CRC failed\')\n    original=json.loads(z.read(\'manifest.json\'));source={e[\'file\']:e for e in original[\'files\']}\n    assert original[\'experiment\']==24\n    for name in SOURCE_MEMBERS:\n     raw=z.read(name);assert len(raw)==source[name][\'bytes\']and hashlib.sha256(raw).hexdigest()==source[name][\'sha256\']\n    extract_checked(z,SOURCE_MEMBERS,root)\n    reports.append({\'source\':\'original24\',\'sha256\':sha,\'files\':len(SOURCE_MEMBERS)})\n   elif sha==STARTER_SHA:\n    entries=verify_manifest(z,25,\'experiment25-starter\')\n    extract_checked(z,{e[\'file\']:e[\'file\']for e in entries},root)\n    reports.append({\'source\':\'compact25-starter\',\'sha256\':sha,\'files\':len(entries)})\n   else:\n    entries=verify_manifest(z,25)\n    if \'protocol.json\'not in z.namelist():raise ValueError(\'Early snapshot is not a training-boundary resume ZIP; supply starter25 or original24\')\n    protocol=json.loads(z.read(\'protocol.json\'))\n    if protocol!={\'protocol\':run.PROTOCOL,\'code_sha256\':code_sha}:raise ValueError(\'Resume code/protocol changed\')\n    extract_checked(z,{e[\'file\']:e[\'file\']for e in entries},root)\n    reports.append({\'source\':\'resume25\',\'sha256\':sha,\'files\':len(entries)})\n validate_inputs(root,check_tensors=True)\n protocol={\'protocol\':run.PROTOCOL,\'code_sha256\':code_sha}\n if(root/\'protocol.json\').exists():assert json.loads((root/\'protocol.json\').read_text())==protocol\n else:core.write_json(root/\'protocol.json\',protocol)\n return reports\n\n\ndef verified_local_copy(paths,cache):\n cache=Path(cache);cache.mkdir(parents=True,exist_ok=True);copied=[]\n for source in map(Path,paths):\n  if not source.is_file():raise FileNotFoundError(f\'Input file missing: {source}\')\n  digest=core.sha_file(source);target=cache/source.name\n  if target.exists()and core.sha_file(target)==digest:\n   copied.append(target);continue\n  temporary=target.with_suffix(target.suffix+\'.tmp\')\n  shutil.copyfile(source,temporary)\n  if core.sha_file(temporary)!=digest:raise ValueError(\'Drive/local copy SHA mismatch\')\n  temporary.replace(target);copied.append(target)\n return copied\n'}
for name,source in MODULE_SOURCES.items():(MODULE_DIR/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(MODULE_DIR.resolve()))
import recipe25_core as core,recipe25_exec as numeric,recipe25_data as data,recipe25_run as recipe,recipe25_input as inputs
for module in (core,numeric,data,recipe,inputs):importlib.reload(module)
import torch,numpy as np,tokenizers,pyarrow,sympy
assert torch.cuda.is_available()and torch.cuda.device_count()==1,'Нужна однаGPU'
assert 'T4'in torch.cuda.get_device_name(0),'Протокол рассчитан наT4'
assert tokenizers.__version__=='0.23.2'
torch.set_num_threads(2);torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
torch.set_float32_matmul_precision('highest');device=torch.device('cuda')
CODE_SHA=hashlib.sha256(json.dumps(MODULE_SOURCES,sort_keys=True).encode()).hexdigest()
RUN_DIR=Path('hw02-lr-dropout-after-normalized-embeddings-results');RUN_DIR.mkdir(exist_ok=True)
ARCHIVE=RUN_DIR.with_suffix('.zip');FUSED=core.fused_adamw_available(device)
# 'replay': only if originalR5 is selected, reproduce its training for a full state.
# 'weights_only': export originalR5 without optimizer; new L/D runs always have full state.
CONTROL_STATE_POLICY='replay'
BACKUP_EACH_BOUNDARY_TO_DRIVE=False
DRIVE_RESULTS_DIR=Path('/content/drive/MyDrive/nlp-hw02/results')
runtime={'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,'CUDA':torch.version.cuda,
 'tokenizers':tokenizers.__version__,'pyarrow':pyarrow.__version__,'sympy':sympy.__version__,
 'SDPA_enabled_flags':core.sdpa_flags(),'fusedAdamW_available':FUSED,'TF32':False}
print('RUNTIME',json.dumps(runtime,ensure_ascii=False))
print(subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total,memory.free','--format=csv'],capture_output=True,text=True).stdout)
if str(torch.__version__).split('+')[0]!='2.11.0':print('NOTE:24 used torch2.11.0; exact initialized tensors supplied, control exec/CE gate remains mandatory.')
with torch.device('meta'):parameter_model=core.SimpleTransformer(**core.MODEL_SPEC)
print('PARAMETERS',sum(p.numel()for p in parameter_model.parameters()));del parameter_model
print('PROTOCOL BEFORE RUNS',json.dumps(recipe.PROTOCOL,ensure_ascii=False,indent=2))
def persist():
 result['elapsed_session_since_first_cell']=time.monotonic()-_RECIPE25_FIRST_CELL
 if 'SESSION_ACTIVE_STARTED'in globals():result['active_session_seconds']=time.monotonic()-SESSION_ACTIVE_STARTED
 core.write_json(RUN_DIR/'results.json',result)
def pack_boundary(folder,archive):
 persist();record=recipe.pack_results(folder,archive);print('ZIP',record,flush=True)
 if BACKUP_EACH_BOUNDARY_TO_DRIVE:
  inputs.verified_local_copy([archive],DRIVE_RESULTS_DIR)
 return record
def snapshot():return pack_boundary(RUN_DIR,ARCHIVE)
def model_path(rid):return RUN_DIR/('controls/R5/model.pt'if rid=='R5'else f'runs/{rid}/checkpoint.pt')

In [ ]:
INPUT_MODE='auto'  # auto /drive /paths /upload; auto:reuse, otherwise Drive
INPUT_PATHS=['/content/drive/MyDrive/nlp-hw02/25-start-from-24-r5-and-fixed-plan.zip']
FORCE_IMPORT=False
transfer_start=time.monotonic();input_paths=[]
try:
 inputs.validate_inputs(RUN_DIR);already_ready=True
except (ValueError,FileNotFoundError,AssertionError):already_ready=False
if not already_ready or FORCE_IMPORT:
 if INPUT_MODE in {'auto','drive'}:
  from google.colab import drive
  drive.mount('/content/drive')
  input_paths=inputs.verified_local_copy(INPUT_PATHS,'hw02-recipe25-input')
 elif INPUT_MODE=='paths':input_paths=inputs.verified_local_copy(INPUT_PATHS,'hw02-recipe25-input')
 elif INPUT_MODE=='upload':
  from google.colab import files
  uploaded=files.upload();folder=Path('hw02-recipe25-upload');folder.mkdir(exist_ok=True)
  for name,raw in uploaded.items():
   p=folder/Path(name).name;p.write_bytes(raw);input_paths.append(p)
  del uploaded
 else:raise ValueError('Unknown INPUT_MODE')
upload_report=inputs.import_inputs(input_paths,RUN_DIR,CODE_SHA)
TOKENIZER_PATH=RUN_DIR/'tokenizer.json';INITIAL=RUN_DIR/'initial.pt'
previous=json.loads((RUN_DIR/'results.json').read_text())if(RUN_DIR/'results.json').exists() else None
result=previous if previous and previous.get('status')!='partial_before_setup'else{
 'experiment':25,'protocol':recipe.PROTOCOL,'status':'inputs_verified','runs':{},'controls':{},'sessions':[]}
assert result['experiment']==25 and result['protocol']==recipe.PROTOCOL
if 'fused_adamw'in result:assert result['fused_adamw']==FUSED,'Optimizer mode changed across resume'
else:result['fused_adamw']=FUSED
if 'control_state_policy'in result:assert result['control_state_policy']==CONTROL_STATE_POLICY,'Control state policy changed across resume'
else:result['control_state_policy']=CONTROL_STATE_POLICY
result['sessions'].append({'runtime':runtime,'inputs':upload_report,'transfer_seconds':time.monotonic()-transfer_start})
core.write_json(RUN_DIR/'runtime.json',result['sessions']);SESSION_ACTIVE_STARTED=time.monotonic();persist()
print('INPUTS SHA VERIFIED BEFORE DATA',upload_report)
if BACKUP_EACH_BOUNDARY_TO_DRIVE:
 from google.colab import drive
 drive.mount('/content/drive')

## Данные и план24

Pinned HF revision прежняя; download толькоtrain0–2 иtest1919. Train500k/old1024/fresh-exclusion hashes должны совпасть с24. Reference-фильтрацияtrain выключена. Новыйheldout не выбирается: используется exactJSON24 и проверка SHA/уникальности/исключений. Fresh только проверяется как множество исключения, без tokenization/генерации/оценки.

Песочница прежняя: AST-фильтр опасных names/imports, ограниченный пул, RLIMIT_CPU и Linux RLIMIT_AS, wall-timeout и parent per-task timeout/restart при killedworker. Bool→int, конечное число; reference не исправляется математически. Числа сравниваются `math.isclose(rel_tol=1e-6,abs_tol=1e-6)`.

In [ ]:
try:
 result['status']='data_preparation';persist()
 tokenizer,tokenbytes=core.init_tokenizer(TOKENIZER_PATH)
 cache,old,test,paths,info,fresh_questions=data.prepare_data('hw02-data',tokenizer)
 assert old==json.loads((RUN_DIR/'old1024.json').read_text())
 new=json.loads((RUN_DIR/'heldout4k.json').read_text())
 assert core.sha_file(RUN_DIR/'heldout4k.json')==recipe.PROTOCOL['heldout4k_file_sha256']
 forbidden={r['question']for r in old+test}|fresh_questions
 assert len(new)==len({r['question']for r in new})==4096
 assert data.data_hash(new)==recipe.PROTOCOL['heldout4k_pairs_sha256']
 assert not({r['question']for r in new}&forbidden)
 assert all(data.eligible(r)and data.is_validation(r['question'])for r in new)
 del fresh_questions,forbidden
 sets={'heldout4k':new,'old1024':old};refs={}
 for split,rows in sets.items():
  refs[split]=numeric.evaluate_programs([r['code']for r in rows])
  expected=4096 if split=='heldout4k' else 991
  assert refs[split]['summary']['complete']and refs[split]['summary']['statuses'].get('numeric')==expected
  saved_reference=json.loads((RUN_DIR/f'{split}-reference-execution.json').read_text())
  agreement=numeric.compare_exec(saved_reference,refs[split])
  assert agreement['matches']==agreement['denominator']==expected,'Reference execution changed from24'
  core.write_json(RUN_DIR/f'{split}-reference-execution.json',refs[split]);print('REFERENCE',split,refs[split]['summary'])
 plan=np.load(RUN_DIR/'batch-plan.npy',allow_pickle=False)
 assert core.sha_file(RUN_DIR/'batch-plan.npy')==recipe.PROTOCOL['batch_plan_file_sha256']
 assert plan.shape==(6250,128)and plan.dtype==np.int32 and int(plan.max())<500000
 sl=np.array([len(x[0])for x in cache]);tl=np.array([len(x[1])+1 for x in cache])
 counts={'pairs':int(plan.size),'source_tokens':int(sl[plan].sum()),'target_tokens':int(tl[plan].sum())}
 assert counts=={'pairs':800000,'source_tokens':38706084,'target_tokens':52797347};del sl,tl
 initial=recipe.load_model(INITIAL,torch.device('cpu'),scaled_tokens=True)
 assert recipe.state_digest(initial.state_dict())==recipe.PROTOCOL['normalized_initial_tensor_sha256'];del initial
 info.update(heldout4k_pairs_sha256=data.data_hash(new),heldout4k_file_sha256=core.sha_file(RUN_DIR/'heldout4k.json'),
  heldout4k_examples=4096,counts_per_launch=counts,initial_tensor_sha256=recipe.PROTOCOL['normalized_initial_tensor_sha256'])
 result['data']=info;core.write_json(RUN_DIR/'data.json',info);result['status']='data_ready';snapshot()
 print('FIXED PRESENTATIONS / USEFUL TOKENS',counts)
except BaseException as error:
 result['status']='setup_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## Переоценка контроляR5

ГенерацияFP16 sortedb512/cap512, CEFP32 b32. Проверки1602±3 иCE±1e-4 выполняются до новых6250-step запусков. На повторе сессии переиспользуется только завершённая **оценка25**, привязанная кSHA весов ипротокола; метрики исходного24 остаются отдельными source-files.

In [ ]:
try:
 result['status']='control_R5';persist();folder=RUN_DIR/'controls/R5'
 if(folder/'metrics.json').exists():
  CONTROL=json.loads((folder/'metrics.json').read_text())
  assert CONTROL['model_sha256']==core.sha_file(folder/'model.pt')
  assert CONTROL['evaluation_protocol_sha256']==recipe.protocol_hash()
 else:
  model=recipe.load_model(folder/'model.pt',device).eval().half()
  src=core.source_batch(tokenizer,[old[0]['question']],device)
  result['observed_SDPA_control']=core.profile_sdpa(lambda:core.generate_preallocated(model,src,8),device)
  print('OBSERVED SDPA',result['observed_SDPA_control']);del model,src;gc.collect();torch.cuda.empty_cache()
  CONTROL=recipe.evaluate_one(folder/'model.pt',recipe.run_spec('R5',.002),sets,refs,TOKENIZER_PATH,device,folder)
 gate=recipe.verify_control(CONTROL);core.write_json(RUN_DIR/'control-reproduction.json',gate)
 CONTROL['quality_eligible']=True;result['controls']['R5']=CONTROL
 result['status']='control_passed';snapshot();print('R5 REPRODUCTION',gate)
except BaseException as error:
 result['status']='control_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## L1/L2 и условныйL3

Веса каждого нового запуска с нуля; общийnormalizedinitial иplan. LR-победитель первого этапа выбирается среди{L1,R5,L2}; решениеL3 фиксируется доего запуска. Завершённыеtrain/eval сreceipt/SHA не повторяются. ZeroAMPskips/finite обязательны; при непригодном запуске сетка останавливается, без выбора по неполному наборуLR.

In [ ]:
try:
 for rid,lr in [('L1',.001),('L2',.003)]:
  result['status']='running_'+rid;persist()
  result['runs'][rid]=recipe.launch(recipe.run_spec(rid,lr),RUN_DIR,INITIAL,cache,plan,sets,refs,TOKENIZER_PATH,
   device,CONTROL,FUSED,pack_fn=pack_boundary)
  snapshot()
 initial_candidates={'R5':CONTROL,**{rid:result['runs'][rid]for rid in ['L1','L2']}}
 initial_best=recipe.select_best(initial_candidates);l3=recipe.conditional_L3(initial_best)
 result['initial_LR_selection']={'id':initial_best['id'],'peak_lr':initial_best['config']['peak_lr'],'L3_config':l3}
 snapshot();print('L3 DECISION',result['initial_LR_selection'])
 if l3 is not None:
  result['status']='running_L3';persist()
  result['runs']['L3']=recipe.launch(l3,RUN_DIR,INITIAL,cache,plan,sets,refs,TOKENIZER_PATH,
   device,CONTROL,FUSED,pack_fn=pack_boundary)
 else:print('L3 skipped: initial best LR0.002')
 LR_CANDIDATES={'R5':CONTROL,**{rid:m for rid,m in result['runs'].items()if rid in ['L1','L2','L3']}}
 LR_BEST=recipe.select_best(LR_CANDIDATES);BEST_LR=LR_BEST['config']['peak_lr']
 result['best_LR_run']=LR_BEST['id'];result['best_peak_LR']=BEST_LR;result['status']='LR_selected';snapshot()
 print('BEST LR',LR_BEST['id'],BEST_LR)
except BaseException as error:
 result['status']='LR_search_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## D0 при выбранномLR

Толькоdropout0; initial/state/батчи/800k предъявлений/LR schedule/AMP/AdamW прежние. Сравнение — с выбранным LR послеL3, включаяR5 если он победил. Правило допускает максимум20 потерянных совпадений из4096.

In [ ]:
try:
 LR_CANDIDATES={'R5':CONTROL,**{rid:m for rid,m in result['runs'].items()if rid in ['L1','L2','L3']}}
 LR_BEST=recipe.select_best(LR_CANDIDATES);BEST_LR=LR_BEST['config']['peak_lr']
 result['status']='running_D0';persist()
 result['runs']['D0']=recipe.launch(recipe.run_spec('D0',BEST_LR,dropout=0.),RUN_DIR,INITIAL,cache,plan,sets,refs,
  TOKENIZER_PATH,device,CONTROL,FUSED,pack_fn=pack_boundary)
 result['status']='runs_evaluated';snapshot()
except BaseException as error:
 result['status']='D0_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## Решение, checkpoint иtest1919

Печатаются правила/таблица/McNemar/length bins и фактический бюджет. Выбранный новый checkpoint содержитFP32model иAdamW/GradScaler/scheduler/RNG. Если победил исходныйR5, действуетполитика состояния изsetup: исходный24optimizer отсутствует. ПриповтореR5 его checkpoint относитсяк **повторному** обучению; исходная иповторная модель/метрики различаютсяв отчёте.

Test1919 — FP16 sortedb512 greedy/cap512, исходный порядок; fulltimer отзагрузки checkpoint до закрытия файлаJSONL, включаяtokenizer/H2D/warmup/sort/restore/write. Download/cache/exec/CE/полныйJSON dump вне timer. Goldtest недоступен. `test-answers.jsonl` — кандидат длябота; схема официальной посылки пока не подтверждена.

In [ ]:
try:
 LR_CANDIDATES={'R5':CONTROL,**{rid:m for rid,m in result['runs'].items()if rid in ['L1','L2','L3']}}
 LR_BEST=recipe.select_best(LR_CANDIDATES);all_metrics={**LR_CANDIDATES,'D0':result['runs']['D0']}
 decision=recipe.decisions(all_metrics,LR_BEST);BEST=all_metrics[decision['best_run']]
 if BEST['id']=='R5' and CONTROL_STATE_POLICY=='replay':
  result['status']='R5_state_replay';persist()
  replay=recipe.launch(recipe.run_spec('R5_replay',.002),RUN_DIR,INITIAL,cache,plan,sets,refs,TOKENIZER_PATH,
   device,CONTROL,FUSED,pack_fn=pack_boundary)
  result['runs']['R5_replay']=replay
  # A new training run owns its own optimizer; never attach these states to original24 model tensors.
  gate=recipe.verify_control(replay)
  original_state=torch.load(model_path('R5'),map_location='cpu',weights_only=True)
  original_tensor_SHA=recipe.state_digest(original_state);del original_state
  decision['R5_state_replay']={'reproduction':gate,'original_control_tensor_sha256':original_tensor_SHA,
   'replay_tensor_sha256':replay['training']['model_tensor_sha256'],
   'tensor_identical_to_original':replay['training']['model_tensor_sha256']==original_tensor_SHA,
   'checkpoint_ownership':'new same-recipe training, not reconstructed original24 optimizer',
   'actual_exec4k':replay['splits']['heldout4k']['exec']['matches'],'actual_CE4k':replay['splits']['heldout4k']['CE']['CE']}
  BEST=replay
 elif BEST['id']=='R5' and CONTROL_STATE_POLICY=='require_choice':
  raise RuntimeError('R5 won but24 has no optimizer/scaler. Set CONTROL_STATE_POLICY=replay or weights_only explicitly, then rerun this reporting cell.')
 elif BEST['id']=='R5' and CONTROL_STATE_POLICY!='weights_only':raise ValueError('Unknown control state policy')
 decision['exported_run']=BEST['id'];decision['full_training_checkpoint']=BEST['id']!='R5'
 result['decision']=decision;core.write_json(RUN_DIR/'decision.json',decision)
 table=[]
 for rid,m in {'R5':CONTROL,**result['runs']}.items():
  if m.get('status')!='evaluated':continue
  s=m['splits'];t=m.get('training',{})
  row={'id':rid,'LR':m['config']['peak_lr'],'dropout':m['config']['dropout'],
   '4k_matches':s['heldout4k']['exec']['matches'],'4k_accuracy':s['heldout4k']['exec']['accuracy'],
   'old991_matches':s['old1024']['exec']['matches'],'CE4k':s['heldout4k']['CE']['CE'],
   'CEold1024':s['old1024']['CE']['CE'],'McNemar_vsR5':s['heldout4k'].get('McNemar_against_R5',{}).get('exact_two_sided_p')}
  if t:row.update(train_min=t['train_seconds']/60,wall_min=t['wall_seconds']/60,pairs=t['counts']['pairs'],
   source_tokens=t['counts']['source_tokens'],target_tokens=t['counts']['target_tokens'],
   pairs_per_s=t['pairs_per_second'],source_tokens_per_s=t['useful_source_tokens_per_second'],
   target_tokens_per_s=t['useful_target_tokens_per_second'],AMP_skips=t['AMP_skips'],peak_reserved_GB=(t['peak_reserved_bytes'] or 0)/1e9)
  table.append(row)
 import pandas as pd
 display(pd.DataFrame(table));print('DECISION',json.dumps(decision,ensure_ascii=False,indent=2))
 for rid,m in {'R5':CONTROL,**result['runs']}.items():print('LENGTH BINS',rid,json.dumps(m['splits']['heldout4k']['length_accuracy'],ensure_ascii=False))
 best_model=model_path(BEST['id'])
 best_record={'run':BEST['id'],'path':str(best_model.relative_to(RUN_DIR)),'sha256':core.sha_file(best_model),
  'full_training_state':BEST['id']!='R5','states':['model','AdamW','GradScaler','scheduler','RNG']if BEST['id']!='R5'else['model'],
  'actual_exec4k':BEST['splits']['heldout4k']['exec']['matches'],'actual_CE4k':BEST['splits']['heldout4k']['CE']['CE']}
 core.write_json(RUN_DIR/'best-checkpoint.json',best_record);print('BEST CHECKPOINT',best_record)
 if(RUN_DIR/'test-time.json').exists():
  test_time=json.loads((RUN_DIR/'test-time.json').read_text())
  assert test_time['run']==BEST['id']and test_time['model_SHA256']==core.sha_file(best_model)
  assert core.sha_file(RUN_DIR/'test-answers.jsonl')==test_time['answers_sha256']
 else:
  examples,test_time=recipe.generate_one(best_model,test,TOKENIZER_PATH,device,RUN_DIR,'test')
  assert[e['question']for e in examples]==[r['question']for r in test]
  test_time.update(run=BEST['id'],model_SHA256=core.sha_file(best_model),under180_seconds=test_time['full_seconds']<180)
  core.write_json(RUN_DIR/'test-time.json',test_time)
 result['test']=test_time;result['table']=table;result['status']='complete';snapshot()
 result['within75_active_session_minutes']=result['active_session_seconds']<=75*60;persist()
 recipe.pack_results(RUN_DIR,ARCHIVE)  # Include the final budget receipt in the local ZIP as well.
 print('TEST FULL WALL',test_time)
 print('BUDGET',{'active_session_minutes':result['active_session_seconds']/60,
  'since_first_cell_minutes':result['elapsed_session_since_first_cell']/60,'within75':result['within75_active_session_minutes'],
  'all_completed_train_minutes':sum(m['training']['train_seconds']for m in result['runs'].values())/60,
  'includes':'preparation/eval/checkpoint/ZIP; final download/optional Drive transfer outside'})
except BaseException as error:
 result['status']='report_or_test_interrupted';result['last_error']=type(error).__name__+': '+str(error)[:500];snapshot();raise

## ТекущийZIP: скачать или сохранитьвDrive послеinterrupt

Эта ячейка работает доsetup ипослепрерывания: толькоупаковка/CRC/SHA. Частьдоsetup ещёнеresume-архив. В полноценномZIP: исходныеR5/initial/tokenizer/план/heldout, каждый завершённыйcheckpoint+генерации/метрики, test1919 и `best-checkpoint.json` с путём выбранногоPT. Trainкорпус исключён; freshrows отсутствуют. СкачиватьвесьZIP длявозобновлениячерезбраузер ненужно: сохранитьвDrive иуказатьегопуть впервойячейке.

ДоT4 провереныCPUtinyFP32/FP16Scaler, scheduler/checkpoint/resume, conditionalLR/dropout rules, order/greedy иarchiveSHA. **Непроверено наT4:** полныйtrain6250×128 икачествоL1/L2/L3/D0, воспроизведениеR5greedy/CE, CUDAFP16/fusedAdamW/SDPA на длинном горизонте, memory/time60–75мин, реальныеColabDrive/UI. CPUпроверки не являются результатом25.

In [ ]:
import json,hashlib,zipfile,shutil
from pathlib import Path
SAVE_TO_DRIVE=False
DOWNLOAD_ZIP=True
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/nlp-hw02/results'
folder=Path('hw02-lr-dropout-after-normalized-embeddings-results');folder.mkdir(exist_ok=True)
if not(folder/'results.json').exists():(folder/'results.json').write_text(json.dumps({'experiment':25,'status':'partial_before_setup'}),encoding='utf-8')
def file_sha(p):
 h=hashlib.sha256()
 with p.open('rb')as f:
  for b in iter(lambda:f.read(1024**2),b''):h.update(b)
 return h.hexdigest()
paths=[p for p in sorted(folder.rglob('*'))if p.is_file()and p.name!='manifest.json'and not p.name.endswith('.tmp')]
manifest={'experiment':25,'files':[{'file':str(p.relative_to(folder)),'bytes':p.stat().st_size,'sha256':file_sha(p)}for p in paths]}
(folder/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
archive=folder.with_suffix('.zip');pending=archive.with_suffix('.zip.tmp')
with zipfile.ZipFile(pending,'w')as z:
 for p in paths+[folder/'manifest.json']:
  z.write(p,arcname=str(p.relative_to(folder)),compress_type=zipfile.ZIP_STORED if p.suffix=='.pt'else zipfile.ZIP_DEFLATED,compresslevel=1)
with zipfile.ZipFile(pending)as z:
 assert z.testzip()is None
 for entry in manifest['files']:
  with z.open(entry['file'])as f:
   h=hashlib.sha256()
   for b in iter(lambda:f.read(1024**2),b''):h.update(b)
  assert h.hexdigest()==entry['sha256']
pending.replace(archive)
print('ZIP',archive.name,archive.stat().st_size,'bytes; status',json.loads((folder/'results.json').read_text()).get('status'))
if SAVE_TO_DRIVE:
 from google.colab import drive
 drive.mount('/content/drive');target=Path(DRIVE_OUTPUT_DIR);target.mkdir(parents=True,exist_ok=True)
 destination=target/archive.name;temporary=destination.with_suffix('.zip.tmp')
 shutil.copyfile(archive,temporary);assert file_sha(temporary)==file_sha(archive);temporary.replace(destination)
 print('DRIVE BACKUP',destination)
if DOWNLOAD_ZIP:
 from google.colab import files
 files.download(str(archive))